# Model Evaluation

In this notebook, we learn various metrics and methods for evaluating machine learning model performance.

## Table of Contents
1. Classification Evaluation Metrics
   - Confusion Matrix
   - Accuracy, Precision, Recall, F1-score
   - ROC Curve and AUC
   - Precision-Recall Curve
2. Multiclass Classification Evaluation
3. Regression Evaluation Metrics
4. Learning Curves

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, load_iris, load_diabetes
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report,
    roc_curve, roc_auc_score, auc,
    precision_recall_curve, average_precision_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.preprocessing import label_binarize

# Visualization settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

## 1. Classification Evaluation Metrics

### 1.1 Confusion Matrix

In [ ]:
# Simple example data
y_true = np.array([1, 0, 1, 1, 0, 1, 0, 0, 1, 1])
y_pred = np.array([1, 0, 1, 0, 0, 1, 1, 0, 1, 1])

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)
print()

# Extract confusion matrix elements
tn, fp, fn, tp = cm.ravel()
print(f"TN (True Negative): {tn}")
print(f"FP (False Positive): {fp} - Type I Error")
print(f"FN (False Negative): {fn} - Type II Error")
print(f"TP (True Positive): {tp}")

In [ ]:
# Confusion matrix visualization
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix', fontsize=14, pad=20)
plt.show()

### 1.2 Accuracy, Precision, Recall, F1-Score

In [ ]:
# Compute each metric
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("=== Classification Evaluation Metrics ===")
print(f"Accuracy: {accuracy:.4f}")
print(f"  - (TP + TN) / (TP + TN + FP + FN)")
print(f"  - Ratio of correct predictions among all predictions\n")

print(f"Precision: {precision:.4f}")
print(f"  - TP / (TP + FP)")
print(f"  - Ratio of actual positives among predicted positives\n")

print(f"Recall (Sensitivity): {recall:.4f}")
print(f"  - TP / (TP + FN)")
print(f"  - Ratio of correctly predicted positives among actual positives\n")

print(f"F1-Score: {f1:.4f}")
print(f"  - 2 * (Precision * Recall) / (Precision + Recall)")
print(f"  - Harmonic mean of precision and recall")

In [ ]:
# Manual calculation for verification
accuracy_manual = (tp + tn) / (tp + tn + fp + fn)
precision_manual = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_manual = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_manual = 2 * precision_manual * recall_manual / (precision_manual + recall_manual) if (precision_manual + recall_manual) > 0 else 0

print("=== Manual Calculation Verification ===")
print(f"Accuracy:  {accuracy_manual:.4f}")
print(f"Precision: {precision_manual:.4f}")
print(f"Recall:    {recall_manual:.4f}")
print(f"F1-Score:  {f1_manual:.4f}")

### 1.3 Classification Evaluation with Real Dataset - Breast Cancer Dataset

In [ ]:
# Load breast cancer dataset
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42
)

# Why: max_iter=10000 prevents convergence warnings on high-dimensional data.
# Logistic Regression is a good evaluation baseline because it produces
# calibrated probabilities needed for ROC and PR curves.
model = LogisticRegression(max_iter=10000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Breast Cancer Classification Results")
print("="*50)
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {cancer.feature_names[:5]}... (total {len(cancer.feature_names)})")

In [ ]:
# Confusion matrix visualization
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Malignant', 'Benign'])
disp.plot(ax=ax, cmap='RdYlGn', values_format='d')
plt.title('Confusion Matrix - Breast Cancer Classification', fontsize=14, pad=20)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")

In [ ]:
# Classification Report
report = classification_report(y_test, y_pred, target_names=['Malignant', 'Benign'])
print("\n=== Classification Report ===")
print(report)

# Also view as dictionary
report_dict = classification_report(y_test, y_pred, target_names=['Malignant', 'Benign'], output_dict=True)
print(f"\nBenign class F1-score: {report_dict['Benign']['f1-score']:.4f}")
print(f"Malignant class Recall: {report_dict['Malignant']['recall']:.4f}")

### 1.4 ROC Curve and AUC

In [ ]:
# Predicted probabilities
y_proba = model.predict_proba(X_test)[:, 1]

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

# ROC curve visualization
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random Classifier (AUC = 0.5)')
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity/Recall)', fontsize=12)
plt.title('ROC Curve - Breast Cancer Classification', fontsize=14, pad=20)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print(f"AUC Score: {roc_auc:.4f}")
print(f"AUC Score (sklearn direct calculation): {roc_auc_score(y_test, y_proba):.4f}")
print("\nAUC Interpretation:")
print("  - 1.0: Perfect classifier")
print("  - 0.5: Random classifier")
print("  - 0.0: Worst classifier")

### 1.5 Precision-Recall Curve

In [ ]:
# Why: PR curve is more informative than ROC on imbalanced datasets — ROC can
# look optimistic when negatives dominate. Average Precision (AP) summarizes
# the PR curve as a single number, analogous to AUC for ROC.
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

# PR curve visualization
plt.figure(figsize=(10, 6))
plt.plot(recall, precision, 'b-', linewidth=2, label=f'PR Curve (AP = {ap:.4f})')
plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curve', fontsize=14, pad=20)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.show()

print(f"Average Precision (AP): {ap:.4f}")
print("\nROC vs PR Curve:")
print("  - ROC: Stable even with imbalanced data, overall performance evaluation")
print("  - PR: More sensitive with imbalanced data, focuses on positive class prediction")

In [ ]:
# ROC and PR curve side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curve
axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'r--', linewidth=2)
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curve', fontsize=14)
axes[0].legend(loc='lower right', fontsize=11)
axes[0].grid(True, alpha=0.3)

# PR Curve
axes[1].plot(recall, precision, 'g-', linewidth=2, label=f'PR (AP = {ap:.4f})')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curve', fontsize=14)
axes[1].legend(loc='best', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Multiclass Classification Evaluation

In [ ]:
# Load Iris dataset (3 classes)
iris = load_iris()
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# Model training
model_iris = LogisticRegression(max_iter=1000, random_state=42)
model_iris.fit(X_train_iris, y_train_iris)
y_pred_iris = model_iris.predict(X_test_iris)

print("Iris Multi-class Classification")
print("="*50)
print(f"Classes: {iris.target_names}")
print(f"Features: {iris.feature_names}")

In [ ]:
# Multiclass confusion matrix
cm_iris = confusion_matrix(y_test_iris, y_pred_iris)
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_iris, display_labels=iris.target_names)
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Multi-class Confusion Matrix - Iris Dataset', fontsize=14, pad=20)
plt.show()

In [ ]:
# Why: Multi-class F1 averaging matters — 'macro' treats all classes equally
# (good for balanced data), 'weighted' accounts for class imbalance,
# 'micro' aggregates globally (equivalent to accuracy for single-label).
print("=== Multi-class Classification Metrics ===")
print(f"Accuracy: {accuracy_score(y_test_iris, y_pred_iris):.4f}\n")

# Various averaging methods for F1-score
f1_macro = f1_score(y_test_iris, y_pred_iris, average='macro')
f1_weighted = f1_score(y_test_iris, y_pred_iris, average='weighted')
f1_micro = f1_score(y_test_iris, y_pred_iris, average='micro')

print(f"F1-Score (macro):    {f1_macro:.4f}  - Simple average of F1 for each class")
print(f"F1-Score (weighted): {f1_weighted:.4f}  - Weighted average by class sample count")
print(f"F1-Score (micro):    {f1_micro:.4f}  - Computed from total TP, FP, FN")

In [ ]:
# Classification report
report_iris = classification_report(y_test_iris, y_pred_iris, target_names=iris.target_names)
print("\n=== Classification Report - Iris ===")
print(report_iris)

In [ ]:
# Multiclass ROC curve
y_test_iris_bin = label_binarize(y_test_iris, classes=[0, 1, 2])
y_proba_iris = model_iris.predict_proba(X_test_iris)

plt.figure(figsize=(10, 6))
colors = ['blue', 'red', 'green']

for i, (color, name) in enumerate(zip(colors, iris.target_names)):
    fpr_i, tpr_i, _ = roc_curve(y_test_iris_bin[:, i], y_proba_iris[:, i])
    roc_auc_i = auc(fpr_i, tpr_i)
    plt.plot(fpr_i, tpr_i, color=color, linewidth=2,
             label=f'{name} (AUC = {roc_auc_i:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=2)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Multi-class ROC Curves - Iris Dataset', fontsize=14, pad=20)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## 3. Regression Evaluation Metrics

In [ ]:
# Simple example
y_true_reg = np.array([3.0, -0.5, 2.0, 7.0, 4.5])
y_pred_reg = np.array([2.5, 0.0, 2.0, 8.0, 4.0])

# Compute regression metrics
mae = mean_absolute_error(y_true_reg, y_pred_reg)
mse = mean_squared_error(y_true_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_true_reg, y_pred_reg)

print("=== Regression Evaluation Metrics ===")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"  - On average, predictions deviate from actual by {mae:.4f}\n")

print(f"MSE (Mean Squared Error): {mse:.4f}")
print(f"  - Penalizes large errors more heavily\n")

print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"  - Interpretable in the same units as the target\n")

print(f"R² (Coefficient of Determination): {r2:.4f}")
print(f"  - Range 0~1, closer to 1 is better")
print(f"  - Model explains {r2*100:.1f}% of variance")

In [ ]:
# Manual calculation for verification
print("\n=== Manual Calculation Verification ===")
mae_manual = np.mean(np.abs(y_true_reg - y_pred_reg))
mse_manual = np.mean((y_true_reg - y_pred_reg)**2)
rmse_manual = np.sqrt(mse_manual)
r2_manual = 1 - np.sum((y_true_reg - y_pred_reg)**2) / np.sum((y_true_reg - np.mean(y_true_reg))**2)

print(f"MAE:  {mae_manual:.4f}")
print(f"MSE:  {mse_manual:.4f}")
print(f"RMSE: {rmse_manual:.4f}")
print(f"R²:   {r2_manual:.4f}")

In [ ]:
# Regression evaluation with real dataset - Diabetes Dataset
diabetes = load_diabetes()
X_train_diab, X_test_diab, y_train_diab, y_test_diab = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=42
)

# Train linear regression model
model_reg = LinearRegression()
model_reg.fit(X_train_diab, y_train_diab)
y_pred_diab = model_reg.predict(X_test_diab)

# Evaluate
mae_diab = mean_absolute_error(y_test_diab, y_pred_diab)
mse_diab = mean_squared_error(y_test_diab, y_pred_diab)
rmse_diab = np.sqrt(mse_diab)
r2_diab = r2_score(y_test_diab, y_pred_diab)

print("Diabetes Regression Results")
print("="*50)
print(f"MAE:  {mae_diab:.4f}")
print(f"MSE:  {mse_diab:.4f}")
print(f"RMSE: {rmse_diab:.4f}")
print(f"R²:   {r2_diab:.4f}")
print(f"\nInterpretation: The model explains {r2_diab*100:.1f}% of the target variance.")

In [ ]:
# Actual vs Predicted visualization
plt.figure(figsize=(10, 6))
plt.scatter(y_test_diab, y_pred_diab, alpha=0.6, edgecolors='k', s=80)
plt.plot([y_test_diab.min(), y_test_diab.max()], 
         [y_test_diab.min(), y_test_diab.max()], 
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual', fontsize=12)
plt.ylabel('Predicted', fontsize=12)
plt.title(f'Actual vs Predicted (R² = {r2_diab:.4f})', fontsize=14, pad=20)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Residual analysis
residuals = y_test_diab - y_pred_diab

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Residual plot
axes[0].scatter(y_pred_diab, residuals, alpha=0.6, edgecolors='k', s=80)
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Residuals', fontsize=12)
axes[0].set_title('Residual Plot', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residuals', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Residuals Distribution', fontsize=14)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Residual mean: {residuals.mean():.4f} (should be close to 0)")
print(f"Residual std: {residuals.std():.4f}")

## 4. Learning Curve

In [ ]:
# Why: Learning curves diagnose bias vs variance. If both curves plateau low,
# the model underfits (high bias). If training is high but validation is low,
# the model overfits (high variance). Convergence means a good fit.
train_sizes, train_scores, val_scores = learning_curve(
    LogisticRegression(max_iter=10000, random_state=42),
    cancer.data, cancer.target,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

# Mean and standard deviation
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

# Learning curve visualization
plt.figure(figsize=(10, 6))
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                 alpha=0.2, color='blue')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                 alpha=0.2, color='orange')
plt.plot(train_sizes, train_mean, 'o-', color='blue', linewidth=2, 
         label='Training Score')
plt.plot(train_sizes, val_mean, 'o-', color='orange', linewidth=2, 
         label='Validation Score')
plt.xlabel('Training Set Size', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve - Breast Cancer Classification', fontsize=14, pad=20)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("Learning curve interpretation:")
print("  - Both curves low: Underfitting (need a more complex model)")
print("  - Training high, validation low: Overfitting (need regularization)")
print("  - Both curves converge: Good fit")

## 5. Metric Selection Guide

In [ ]:
# Comprehensive evaluation functions
def evaluate_classification(y_true, y_pred, y_proba=None):
    """Comprehensive classification model evaluation"""
    print("=== Classification Evaluation Results ===")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted'):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, average='weighted'):.4f}")
    print(f"F1-Score:  {f1_score(y_true, y_pred, average='weighted'):.4f}")
    if y_proba is not None and len(np.unique(y_true)) == 2:
        print(f"ROC-AUC:   {roc_auc_score(y_true, y_proba):.4f}")

def evaluate_regression(y_true, y_pred):
    """Comprehensive regression model evaluation"""
    print("=== Regression Evaluation Results ===")
    print(f"MAE:  {mean_absolute_error(y_true, y_pred):.4f}")
    print(f"MSE:  {mean_squared_error(y_true, y_pred):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
    print(f"R²:   {r2_score(y_true, y_pred):.4f}")

# Test
print("Breast Cancer model evaluation:")
evaluate_classification(y_test, y_pred, y_proba)

print("\nDiabetes regression model evaluation:")
evaluate_regression(y_test_diab, y_pred_diab)

In [ ]:
# Evaluation metrics summary table
import pandas as pd

metrics_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'MAE', 'MSE', 'R²'],
    'Task': ['Classification', 'Classification', 'Classification', 'Classification', 'Classification', 'Regression', 'Regression', 'Regression'],
    'Range': ['0-1', '0-1', '0-1', '0-1', '0-1', '0-inf', '0-inf', '-inf-1'],
    'Description': [
        'Overall correct ratio',
        'True positives among predicted positives',
        'True positives among actual positives',
        'Harmonic mean of Precision/Recall',
        'Overall classifier performance',
        'Mean absolute error',
        'Mean squared error',
        'Explained variance ratio'
    ]
})

print("\n=== Evaluation Metrics Summary ===")
print(metrics_summary.to_string(index=False))

## Summary

### Classification Metric Selection

1. **Balanced data**: Accuracy, F1-score
2. **Imbalanced data**: Precision, Recall, F1-score, PR-AUC
   - Positive class is important: Emphasize Recall (cancer diagnosis, fraud detection)
   - False positives are costly: Emphasize Precision (spam filter)
3. **Probability prediction quality**: ROC-AUC, PR-AUC
4. **Multiclass**: Macro/Weighted/Micro F1

### Regression Metric Selection

1. **Basic**: MSE, RMSE, MAE
2. **Outlier sensitivity**: MAE (robust), MSE (sensitive)
3. **Relative error**: R²
4. **Model comparison**: R² (normalized to 0~1 range)